<h1>Notebook utile solo alla creazione dei dati da usare per YOLO</h1>

In [2]:
import os
from pathlib import Path
import shutil
from collections import defaultdict

from PIL import Image  # pip install pillow se manca
from python_file.dirPath import csvDir, txtDir, imageDir, dayTest, fogTest, nightTest


In [9]:
# ================== PARAMETRI DA MODIFICARE ==================

# Cartella di input che contiene:
#  - le immagini (es: 5.png, 6.png, ...)
#  - il file "annotation.txt"
INPUT_DIR = Path(r"/home/marco/progetti/MachineLearning_Unict/yolo_image/night")

# Cartella di output (root del dataset YOLO)
OUTPUT_DIR = Path(r"/home/marco/progetti/MachineLearning_Unict/yolo_image")

# Offset per rinominare le immagini:
# es. OFFSET = 1000, immagine "5.png" -> "1005.png" / "1005.txt"
OFFSET = 2000

# Se True, ogni VAL_EVERY_N-esima immagine va in "val" anziché "train"
USE_VAL_SPLIT = True
VAL_EVERY_N = 5  # vale solo se USE_VAL_SPLIT è True

# Nome cartelle (puoi cambiarli se vuoi, ma YOLO di solito usa "images" / "labels")
IMAGES_DIR_NAME = "images"
LABELS_DIR_NAME = "labels"

# Se hai UNA sola classe "traffic_sign"
CLASS_ID = 0

# Nome file annotazioni nella cartella di input
ANNOTATION_FILE_NAME = "annotations.txt"


In [10]:
def parse_annotations(annotation_path: Path):
    """
    Legge annotation.txt nel formato:
    img_name x_min y_min x_max y_max class_name
    e restituisce un dict: {img_name: [(x_min, y_min, x_max, y_max, class_id), ...]}
    """
    annotations = defaultdict(list)
    if not annotation_path.exists():
        raise FileNotFoundError(f"File annotazioni non trovato: {annotation_path}")

    with annotation_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split()
            if len(parts) < 6:
                # riga malformata, la saltiamo
                print(f"Riga ignorata (pochi campi): {line}")
                continue

            img_name = parts[0]
            try:
                x_min = float(parts[1])
                y_min = float(parts[2])
                x_max = float(parts[3])
                y_max = float(parts[4])
            except ValueError:
                print(f"Riga ignorata (coordinate non numeriche): {line}")
                continue

            # Hai una sola classe -> CLASS_ID fisso (es. 0)
            class_id = CLASS_ID
            annotations[img_name].append((x_min, y_min, x_max, y_max, class_id))

    return annotations


def convert_box_to_yolo(x_min, y_min, x_max, y_max, img_w, img_h):
    """
    Converte box in pixel (x_min, y_min, x_max, y_max)
    in formato YOLO normalizzato (x_center, y_center, width, height).
    """
    x_center = (x_min + x_max) / 2.0
    y_center = (y_min + y_max) / 2.0
    width = x_max - x_min
    height = y_max - y_min

    x_center_norm = x_center / img_w
    y_center_norm = y_center / img_h
    width_norm = width / img_w
    height_norm = height / img_h

    return x_center_norm, y_center_norm, width_norm, height_norm


def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def main():
    input_dir = INPUT_DIR
    output_dir = OUTPUT_DIR

    # Controlli base
    if not input_dir.exists():
        raise FileNotFoundError(f"Cartella di input non trovata: {input_dir}")

    annotation_path = input_dir / ANNOTATION_FILE_NAME
    annotations = parse_annotations(annotation_path)

    # Directory di output
    train_images_dir = output_dir / IMAGES_DIR_NAME / "train"
    train_labels_dir = output_dir / LABELS_DIR_NAME / "train"
    val_images_dir = output_dir / IMAGES_DIR_NAME / "val"
    val_labels_dir = output_dir / LABELS_DIR_NAME / "val"

    ensure_dir(train_images_dir)
    ensure_dir(train_labels_dir)
    if USE_VAL_SPLIT:
        ensure_dir(val_images_dir)
        ensure_dir(val_labels_dir)

    processed_count = 0

    # Itera su tutte le immagini che compaiono in annotation.txt
    for img_name, boxes in sorted(annotations.items()):
        img_path = input_dir / img_name
        if not img_path.exists():
            print(f"Immagine {img_name} non trovata, salto.")
            continue

        # Determina se va in train o val
        subset = "train"
        if USE_VAL_SPLIT:
            # es: una immagine su VAL_EVERY_N in val
            if (processed_count + 1) % VAL_EVERY_N == 0:
                subset = "val"

        if subset == "train":
            img_out_dir = train_images_dir
            lbl_out_dir = train_labels_dir
        else:
            img_out_dir = val_images_dir
            lbl_out_dir = val_labels_dir

        # Legge dimensioni immagine
        with Image.open(img_path) as im:
            img_w, img_h = im.size

        # Nuovo nome con offset
        stem = img_path.stem  # "5" da "5.png"
        suffix = img_path.suffix  # ".png"

        try:
            orig_index = int(stem)
        except ValueError:
            # Se il nome non è numerico, usa un contatore progressivo
            orig_index = processed_count + 1

        new_index = orig_index + OFFSET
        new_img_name = f"{new_index}{suffix}"
        new_lbl_name = f"{new_index}.txt"

        # Copia immagine
        dest_img_path = img_out_dir / new_img_name
        shutil.copy2(img_path, dest_img_path)

        # Crea file label YOLO
        dest_lbl_path = lbl_out_dir / new_lbl_name
        with dest_lbl_path.open("w", encoding="utf-8") as f:
            for (x_min, y_min, x_max, y_max, class_id) in boxes:
                x_c, y_c, w, h = convert_box_to_yolo(
                    x_min, y_min, x_max, y_max, img_w, img_h
                )
                # Riga YOLO: class_id x_center y_center width height
                f.write(f"{int(class_id)} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")

        processed_count += 1

    print(f"Fatto. Immagini processate: {processed_count}")
    print(f"Train images: {len(list(train_images_dir.glob('*')))}")
    print(f"Train labels: {len(list(train_labels_dir.glob('*')))}")
    if USE_VAL_SPLIT:
        print(f"Val images: {len(list(val_images_dir.glob('*')))}")
        print(f"Val labels: {len(list(val_labels_dir.glob('*')))}")

# Esegui la conversione
main()

Fatto. Immagini processate: 48
Train images: 201
Train labels: 201
Val images: 48
Val labels: 48
